# M6.3 One-Dimensional FDTD: Pulses, Interfaces and Energy

Book pointer: chapter 17, section 17.2 (Maxwell's equations). The wave equation, plane waves and the Yee method are taught in primers M6.0c and M6.0d.

## 1. Recall

Answer on paper before running any cell.

1. Write the two 1D Yee updates for a wave with $E_y$ on integer nodes and $H_z$ on half nodes. Which Maxwell equation does each come from, and where does $\epsilon$ enter?
2. State the Courant condition in 1D. What happens to the numerical solution if it is violated, and what is special about $S = c\,\Delta t/\Delta x = 1$?
3. A plane wave in vacuum meets a dielectric of refractive index $n$ at normal incidence. Write the amplitude reflection and transmission coefficients $r$ and $t$, and the reflectance and transmittance $R$ and $T$. Why is $T \neq t^2$?
4. Write the Poynting vector of a plane wave travelling along $+x$ with $E$ along $y$. What is its time integral through a plane equal to?
5. Define the skin depth $\delta$ of a good conductor. How does it depend on frequency and conductivity, and what is the general attenuation constant when the conductor is not good?

## 2. Principle to algorithm

Faraday's law and the Ampere-Maxwell law for a wave moving along $x$ with $\mathbf{E} = E_y\hat{\mathbf{y}}$ and $\mathbf{H} = H_z\hat{\mathbf{z}}$ are
$$\mu_0\frac{\partial H_z}{\partial t} = -\frac{\partial E_y}{\partial x}, \qquad \epsilon\frac{\partial E_y}{\partial t} = -\frac{\partial H_z}{\partial x} - \sigma E_y.$$
The Yee grid stores $E_y$ on the nodes $x_i = i\,\Delta x$ and $H_z$ on the half nodes $x_{i+1/2}$, and advances $H_z$ half a time step out of phase with $E_y$. Each spatial derivative is then a central difference between two neighbours, and each time derivative a central difference between two half steps, so the scheme is second order in both. The loss term $\sigma E_y$ is averaged between the old and new time levels, which keeps the update stable for any conductivity.

`fdtd.fdtd_1d` runs this loop. Materials are arrays on the $E$ nodes: `eps_r` enters the $E$ update and `sigma` its loss. A source is either a soft source, a function of time added to $E_y$ at one node every step, which radiates in both directions, or an initial field. A one-way pulse needs both $E_y$ and $H_z$ at $t = 0$ with $H_z = E_y/\eta_0$, and because $H_z$ lives half a step earlier, at $t = -\Delta t/2$, its profile is shifted forward by $c\,\Delta t/2$. The two ends are either Mur absorbing boundaries, which let a wave leave as if the grid continued, or perfect conductors with $E_y = 0$.

Probes record $E_y$ at chosen nodes every step, together with $H_z$ interpolated to the same place and the same time: the average of the two neighbouring half nodes, each averaged between two half steps. With $E$ and $H$ co-located, $S_x = E_yH_z$ is the Poynting flux through that plane, and its time integral is the energy per unit area that crossed it. The time step is $\Delta t = S\,\Delta x/c$ with Courant number $S \le 1$; the library refuses $S > 1$.

### Animation

The Yee grid as two rows of bars: electric field on the nodes and magnetic field on the half nodes. The update alternates, first every $H$ from the two $E$ values beside it, then every $E$ from the two $H$ values beside it, and a Gaussian pulse moves to the right. In the second part the same loop runs with Courant number 1.02: within a few dozen steps the shortest wavelength on the grid grows without bound, and the readout of the largest $|E|$ explodes.

In [ ]:
from emlib import show
show.animation("m6_waves", "YeeLeapfrog")

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import fdtd as fd
from emlib.fields import EPS0
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (7, 4)
C0, ETA0 = fd.C0, fd.ETA0
dx = 1e-3                                                  # 1 mm cells throughout


def one_way_pulse(n_cells, S, profile):
    # E_y at t = 0 on the nodes and H_z at t = -dt/2 on the half nodes for a pulse moving along +x
    dt = S * dx / C0
    x = np.arange(n_cells + 1) * dx
    xh = (np.arange(n_cells) + 0.5) * dx
    return profile(x), profile(xh + C0 * dt / 2) / ETA0, dt


def peak_time(t, y):
    # time of the maximum of y, refined by a parabola through the three samples around it
    i = np.argmax(y)
    a, b, c = y[i - 1], y[i], y[i + 1]
    return t[i] + 0.5 * (a - c) / (a - 2 * b + c) * (t[1] - t[0])

### 3.1 The pulse moves at $c$, and the Courant number controls its shape

A one-way Gaussian pulse of $1/e$ half-width 15 cells starts at $x = 0.2\,$m and passes two probes 0.8 m apart. The speed is the probe separation over the delay between the two peaks. At $S = 1$ the Yee scheme is exact in vacuum: the numerical dispersion relation $\sin(\omega\Delta t/2) = S\sin(k\Delta x/2)$ reduces to $\omega = ck$, every Fourier component moves at $c$, and the pulse arrives unchanged. Below $S = 1$, short wavelengths travel slightly slower than $c$, so the pulse lags by a small amount, loses a little height and leaves a ripple behind it. The effect shrinks as the pulse is resolved by more cells.

In [ ]:
n_cells = 1600
profile = lambda s: np.exp(-((s - 0.2) / 0.015) ** 2)
fig, ax = plt.subplots()
speed = {}
for S in (1.0, 0.5, 0.25):
    E0, H0, dt = one_way_pulse(n_cells, S, profile)
    run = fd.fdtd_1d(n_cells, dx, int(1200 / S), courant=S, Ey0=E0, Hz0=H0, probes=[400, 1200])
    t, pe = run["t"], run["probe_E"]
    speed[S] = 0.8 / (peak_time(t, pe[:, 1]) - peak_time(t, pe[:, 0])) / C0
    print(f"S = {S:4.2f}: speed / c - 1 = {speed[S] - 1:+.1e}   peak at the far probe {pe[:, 1].max():.4f}   "
          f"most negative ripple {pe[:, 1].min():+.1e}")
    k = np.argmax(pe[:, 1])
    window = slice(k - int(60 / S), k + int(60 / S))
    ax.plot((t[window] - t[k]) * C0 / dx, pe[window, 1], label=f"S = {S}")
ax.plot(np.linspace(-60, 60, 400), np.exp(-(np.linspace(-60, 60, 400) / 15) ** 2), "k:", label="starting shape")
ax.set_xlabel("(t - t_peak) c / dx   [cells]"); ax.set_ylabel("E_y at x = 1.2 m"); ax.legend()
ax.set_title("after 1 m of travel: exact at S = 1, slightly dispersed below")
plt.show()

The speed error at $S = 0.5$ is below a tenth of a percent for a pulse 15 cells wide, and the lost height is two tenths of a percent. In 2D and 3D, $S = 1$ is not allowed and there is no magic time step, so this dispersion is always present; the remedy is more cells per wavelength.

The limit $S \le 1$ is a stability condition, not an accuracy one. The library refuses $S > 1$, so the cell below writes the bare loop itself, with the update coefficients $c\,\Delta t/\Delta x$ times $\eta_0$ or $1/\eta_0$, and runs the same pulse at $S = 1$ and $S = 1.02$. A $2\,\%$ violation is enough: the shortest wavelength on the grid has a growth factor above one per step, round-off seeds it, and within 150 steps it has grown by ten orders of magnitude.

In [ ]:
for S in (1.0, 1.02):
    x = np.arange(401) * dx
    E = np.exp(-((x - 0.1) / 0.01) ** 2)
    H = np.zeros(400)
    largest = []
    for k in range(1, 301):
        H -= S / ETA0 * (E[1:] - E[:-1])                   # Faraday
        E[1:-1] -= S * ETA0 * (H[1:] - H[:-1])            # Ampere-Maxwell, PEC ends
        if k in (50, 100, 150, 200, 300):
            largest.append(np.abs(E).max())
    print(f"S = {S}: largest |E| after 50, 100, 150, 200, 300 steps:", "  ".join(f"{v:.2g}" for v in largest))

At $S = 1$ the initial Gaussian splits into two halves of height 0.5 moving apart, as a pulse with $E$ but no $H$ must. At $S = 1.02$ the same loop explodes.

### 3.2 Reflection and transmission at a dielectric step

The right half of the grid, from node 800 on, has $\epsilon_r = 4$, so $n = 2$. At normal incidence the fields on both sides must join so that $E_y$ and $H_z$ are continuous, and with $H = nE/\eta_0$ in each medium that gives the Fresnel coefficients
$$r = \frac{n_1 - n_2}{n_1 + n_2} = -\frac13, \qquad t = \frac{2n_1}{n_1 + n_2} = \frac23.$$
The reflected pulse is inverted. Energy conservation reads $R + T = 1$ with $R = r^2$ and $T = (n_2/n_1)\,t^2$: the transmitted wave has a smaller amplitude than the incident one but carries energy in a medium where the same $E$ goes with $n$ times more $H$. Here $R = 1/9$ and $T = 8/9$.

The probe at node 600 sees the incident pulse and, later, the reflected one; the probe at node 1000 sees the transmitted one. The pulse is 30 cells wide in vacuum, so it still spans 15 cells in the dielectric, where it travels at $c/2$ and is squeezed by a factor 2.

In [ ]:
n_cells, S, iface = 1600, 0.5, 800
eps = np.ones(n_cells + 1)
eps[iface:] = 4.0
E0, H0, dt = one_way_pulse(n_cells, S, lambda s: np.exp(-((s - 0.2) / 0.03) ** 2))
run = fd.fdtd_1d(n_cells, dx, 3200, courant=S, eps_r=eps, Ey0=E0, Hz0=H0, probes=[600, 1000], snapshot_every=400)
t = run["t"]
before, after = run["probe_E"][:, 0], run["probe_E"][:, 1]
t_split = t[np.argmax(before)] + 0.15 / C0               # the incident pulse is gone, the reflection not yet back
inc = before[t < t_split].max()
r_meas = before[t > t_split].min() / inc
t_meas = after.max() / inc
print(f"r = {r_meas:+.5f}   (Fresnel -1/3 = {-1/3:+.5f})")
print(f"t = {t_meas:+.5f}   (Fresnel  2/3 = {2/3:+.5f})")
print(f"R + T = r^2 + 2 t^2 = {r_meas**2 + 2 * t_meas**2:.5f}")

x = np.arange(n_cells + 1) * dx
fig, ax = plt.subplots(figsize=(8, 4))
for k, snap in enumerate(run["snapshots"]):
    if k in (0, 2, 4, 7):
        ax.plot(x, snap, label=f"step {(k + 1) * 400}")
ax.axvspan(x[iface], x[-1], color="C0", alpha=0.08, label="eps_r = 4")
ax.set_xlabel("x [m]"); ax.set_ylabel("E_y"); ax.legend(fontsize=8); ax.set_title("a pulse meets a dielectric")
plt.show()

In the snapshots the transmitted pulse is narrower and slower. Both follow from the frequency being unchanged across the interface while the wavelength drops by $n$. The cell measures them: the distance the transmitted peak moves between snapshots 400 steps apart, against the $c\,\Delta t \times 400$ it would cover in vacuum, and the width of the pulse from the second moment of $E_y^2$, before and after.

In [ ]:
def width(xs, y):
    w = y**2
    m = np.sum(xs * w) / np.sum(w)
    return np.sqrt(np.sum((xs - m) ** 2 * w) / np.sum(w))

snaps = run["snapshots"]
peaks = [x[iface + np.argmax(s[iface:])] for s in snaps[4:8]]
v_trans = np.mean(np.diff(peaks)) / (400 * dt)
print(f"transmitted peak positions every 400 steps: {np.round(peaks, 3)} m")
print(f"speed in the dielectric: {v_trans / C0:.4f} c   (1/n = 0.5)")
w_in = width(x, E0)
w_out = width(x[iface:], snaps[7][iface:])
print(f"pulse width (second moment): {w_in * 1e3:.2f} mm incident, {w_out * 1e3:.2f} mm transmitted, ratio {w_in / w_out:.3f}")

### 3.3 A quarter-wave antireflection layer

A thin layer between the two media can cancel the reflection at one design wavelength. The wave reflected at the front of the layer and the wave reflected at its back must be equal in size and opposite in phase. Equal size needs the two interfaces to reflect equally, $(1 - n_c)/(1 + n_c) = (n_c - n_2)/(n_c + n_2)$, which gives $n_c = \sqrt{n_1n_2}$. Opposite phase needs the round trip through the layer to add half a wavelength, so the thickness is a quarter of the wavelength inside the layer, $d = \lambda_0/4n_c$. For $n_2 = 2$ the coating has $n_c = \sqrt2$, $\epsilon_r = 2$.

On the grid the thickness must be a whole number of cells, so the design wavelength is fitted to the layer: $d = 15$ cells gives $\lambda_0 = 4\sqrt2 \times 15 = 84.9$ cells in vacuum, 42 cells in the $n = 2$ medium. The source is a Gaussian-modulated cosine, two wavelengths wide, so its spectrum is a band around $f_0 = c/\lambda_0$. The reflection coefficient at each frequency is the ratio of the Fourier transforms of the reflected and the incident traces at one probe, so one run measures $|r(f)|$ across the whole band. The closed form for a single layer is
$$r(f) = \frac{r_1 + r_2e^{-2i\delta}}{1 + r_1r_2e^{-2i\delta}}, \qquad \delta = \frac{2\pi n_c d}{\lambda}, \quad r_1 = \frac{1 - n_c}{1 + n_c}, \quad r_2 = \frac{n_c - n_2}{n_c + n_2},$$
the sum of all the partial reflections bouncing inside the layer.

In [ ]:
d_cells, iface, probe, n_cells, S = 15, 2000, 1400, 4000, 0.5
lam0 = 4 * np.sqrt(2) * d_cells * dx
f0 = C0 / lam0
packet = lambda s: np.exp(-((s - 0.6) / (2 * lam0)) ** 2) * np.cos(2 * np.pi * (s - 0.6) / lam0)


def reflection_spectrum(coated):
    eps = np.ones(n_cells + 1)
    eps[iface:] = 4.0
    if coated:
        eps[iface - d_cells:iface] = 2.0
    E0, H0, dt = one_way_pulse(n_cells, S, packet)
    run = fd.fdtd_1d(n_cells, dx, 6800, courant=S, eps_r=eps, Ey0=E0, Hz0=H0, probes=[probe])
    t, a = run["t"], run["probe_E"][:, 0]
    t_split = ((probe * dx - 0.6) + (iface - probe) * dx) / C0      # halfway between incident and reflected arrivals
    freq = np.fft.rfftfreq(len(t), dt)
    r_f = np.fft.rfft(np.where(t > t_split, a, 0.0)) / np.fft.rfft(np.where(t < t_split, a, 0.0))
    return freq, r_f

freq, r_bare = reflection_spectrum(False)
_, r_coat = reflection_spectrum(True)
k0 = np.argmin(np.abs(freq - f0))
band = (freq > 0.7 * f0) & (freq < 1.3 * f0)
nc, n2 = np.sqrt(2.0), 2.0
r1, r2 = (1 - nc) / (1 + nc), (nc - n2) / (nc + n2)
phase = np.exp(-2j * 2 * np.pi * nc * d_cells * dx * freq[band] / C0)
r_theory = (r1 + r2 * phase) / (1 + r1 * r2 * phase)
print(f"design frequency f0 = {f0 / 1e9:.3f} GHz (lambda0 = {lam0 * 1e3:.1f} mm)")
print(f"|r| at f0: bare {abs(r_bare[k0]):.4f}, coated {abs(r_coat[k0]):.5f}")
print(f"reflectance at f0, coated / bare: {abs(r_coat[k0])**2 / abs(r_bare[k0])**2:.1e}")
print(f"largest |simulated - closed form| for the coated layer over 0.7 to 1.3 f0: {np.abs(np.abs(r_coat[band]) - np.abs(r_theory)).max():.4f}")

fig, ax = plt.subplots()
ax.plot(freq[band] / f0, np.abs(r_bare[band]), label="bare interface (FDTD)")
ax.plot(freq[band] / f0, np.abs(r_coat[band]), label="quarter-wave layer (FDTD)")
ax.plot(freq[band] / f0, np.abs(r_theory), "k:", label="single-layer closed form")
ax.axhline(1 / 3, color="grey", lw=0.5)
ax.set_xlabel("f / f0"); ax.set_ylabel("|r|"); ax.legend(); ax.set_title("one run measures the whole band")
plt.show()

The null sits at the design frequency to within the frequency resolution of the run, and the reflection rises on either side as the layer stops being a quarter wavelength thick. Numerical dispersion makes the wavelength on the grid very slightly shorter than $c/f$, which would shift the null by a fraction of a percent; with 42 or more cells per wavelength in the slowest medium that shift is not visible here.

### 3.4 A lossy slab

From node 1000 on the medium has a conductivity $\sigma$. A continuous wave from a soft source at node 200 enters it and decays as $e^{-\alpha x}$. For a plane wave in a medium with $\epsilon_0$, $\mu_0$ and $\sigma$ the attenuation constant is
$$\alpha = \omega\sqrt{\frac{\mu_0\epsilon_0}{2}}\left[\sqrt{1 + \left(\frac{\sigma}{\omega\epsilon_0}\right)^2} - 1\right]^{1/2},$$
which for a good conductor, $\sigma \gg \omega\epsilon_0$, tends to $1/\delta$ with the skin depth $\delta = \sqrt{2/\omega\mu_0\sigma}$. The cell chooses a wavelength of 400 cells and sets $\sigma$ so that $\delta = 20$ cells. Then $\sigma/\omega\epsilon_0 \approx 20$: a fairly good conductor, but not so good that the skin-depth formula is exact. The source ramps up smoothly to avoid a sharp front; after the steady state is reached, the envelope is the largest $|E_y|$ at each node over one period, and its logarithm inside the slab is fitted with a straight line.

In [ ]:
n_cells, S, slab = 2000, 0.5, 1000
dt = S * dx / C0
lam = 400 * dx
f = C0 / lam
w = 2 * np.pi * f
delta = 20 * dx
sigma = 2 / (w * MU0 * delta**2)
ratio = sigma / (w * EPS0)
alpha = w * np.sqrt(MU0 * EPS0 / 2) * np.sqrt(np.sqrt(1 + ratio**2) - 1)
print(f"sigma = {sigma:.4f} S/m,  sigma / (omega eps0) = {ratio:.2f}")
print(f"alpha exact = {alpha:.3f} 1/m,  good-conductor 1/delta = {1 / delta:.3f} 1/m")

sig = np.zeros(n_cells + 1)
sig[slab:] = sigma
period = int(round(1 / (f * dt)))                          # 800 steps
cw = lambda tt: (1 - np.exp(-(tt * f / 3) ** 2)) * np.sin(w * tt)
run = fd.fdtd_1d(n_cells, dx, 10_000, courant=S, sigma=sig, source=cw, src_index=200, snapshot_every=8)
env = np.abs(np.array(run["snapshots"][-period // 8:])).max(axis=0)
x = np.arange(n_cells + 1) * dx
fit = slice(slab + 2, slab + 80)
alpha_fit = -np.polyfit(x[fit] - x[slab], np.log(env[fit]), 1)[0]
print(f"fitted alpha = {alpha_fit:.3f} 1/m: {alpha_fit / alpha - 1:+.1e} from the exact formula, "
      f"{alpha_fit * delta - 1:+.1e} from the skin-depth formula")

fig, ax = plt.subplots()
ax.semilogy((x - x[slab]) / delta, env, label="FDTD envelope")
xs = np.linspace(0, 6, 50)
ax.semilogy(xs, env[slab] * np.exp(-alpha * xs * delta), "k:", label="exp(-alpha x), exact alpha")
ax.axvline(0, color="grey", lw=0.5)
ax.set_xlim(-3, 6); ax.set_ylim(1e-3, 3)
ax.set_xlabel("depth into the slab / delta"); ax.set_ylabel("|E_y| envelope"); ax.legend()
ax.set_title("standing wave in front, exponential decay inside")
plt.show()

In front of the slab the envelope ripples: the incident and reflected waves form a partial standing wave. Inside it falls as a straight line on the log scale, and the slope agrees with the exact attenuation constant to a few parts in $10^4$, while the skin-depth formula is off by $2.4\,\%$ because $\sigma/\omega\epsilon_0 = 20$ is not infinite.

### 3.5 Energy accounting with the Poynting vector

Back to the dielectric step of section 3.2. The probe before the interface records $S_x = E_yH_z$ for the incident pulse, positive, and later for the reflected one, negative. The probe after the interface records the transmitted pulse. Integrated over time each gives the energy per unit area that crossed the plane, and conservation requires $U_{\text{inc}} = U_{\text{ref}} + U_{\text{trans}}$. The incident energy must also equal the energy of the initial pulse, $\tfrac12\int(\epsilon_0E^2 + \mu_0H^2)\,dx$.

In [ ]:
n_cells, S, iface = 1600, 0.5, 800
eps = np.ones(n_cells + 1)
eps[iface:] = 4.0
E0, H0, dt = one_way_pulse(n_cells, S, lambda s: np.exp(-((s - 0.2) / 0.03) ** 2))
run = fd.fdtd_1d(n_cells, dx, 3600, courant=S, eps_r=eps, Ey0=E0, Hz0=H0, probes=[600, 1000])
t = run["t"]
S_before = run["probe_E"][:, 0] * run["probe_H"][:, 0]
S_after = run["probe_E"][:, 1] * run["probe_H"][:, 1]
t_split = t[np.argmax(run["probe_E"][:, 0])] + 0.15 / C0
U_inc = np.sum(S_before[t < t_split]) * dt
U_ref = -np.sum(S_before[t > t_split]) * dt
U_tr = np.sum(S_after) * dt
U0 = fd.energy_1d(E0, H0, dx)
print(f"energy of the initial pulse:  {U0:.6e} J/m^2")
print(f"incident through x = 0.6 m:   {U_inc:.6e} J/m^2   ratio {U_inc / U0:.5f}")
print(f"reflected: {U_ref / U_inc:.5f} of incident (R = 1/9 = {1/9:.5f})")
print(f"transmitted: {U_tr / U_inc:.5f} of incident (T = 8/9 = {8/9:.5f})")
print(f"(reflected + transmitted) / incident = {(U_ref + U_tr) / U_inc:.6f}")

fig, ax = plt.subplots()
ax.plot(t * 1e9, S_before, label="S_x at x = 0.6 m")
ax.plot(t * 1e9, S_after, label="S_x at x = 1.0 m")
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("t [ns]"); ax.set_ylabel("S_x [W/m^2]"); ax.legend(); ax.set_title("energy in, energy back, energy through")
plt.show()

### Exercise: measure r and t from the probes

Replace the body of `measure_r_t` so that it returns the amplitude reflection and transmission coefficients from two probe traces: `probe_before` in the first medium, which sees the incident pulse before `t_split` and the reflected one after it, and `probe_after` in the second medium. Use the peak of the incident pulse as the reference and keep the sign of the reflection. The check runs a new interface, $\epsilon_r = 2.25$ ($n = 1.5$), and compares with Fresnel.

In [ ]:
def measure_r_t(probe_before, probe_after, t, t_split):
    # replace from here with your own version
    inc = probe_before[t < t_split].max()
    refl = probe_before[t > t_split]
    r = refl[np.argmax(np.abs(refl))] / inc
    tr = probe_after[np.argmax(np.abs(probe_after))] / inc
    return r, tr


n2 = 1.5
eps = np.ones(1601)
eps[800:] = n2**2
E0, H0, dt = one_way_pulse(1600, 0.5, lambda s: np.exp(-((s - 0.2) / 0.03) ** 2))
run = fd.fdtd_1d(1600, dx, 3200, courant=0.5, eps_r=eps, Ey0=E0, Hz0=H0, probes=[600, 1000])
t_split = run["t"][np.argmax(run["probe_E"][:, 0])] + 0.15 / C0
r_x, t_x = measure_r_t(run["probe_E"][:, 0], run["probe_E"][:, 1], run["t"], t_split)
print(f"n = 1.5: r = {r_x:+.5f} (Fresnel {(1 - n2) / (1 + n2):+.5f}),  t = {t_x:+.5f} (Fresnel {2 / (1 + n2):+.5f})")
assert abs(r_x - (1 - n2) / (1 + n2)) < 0.01 * abs((1 - n2) / (1 + n2))
assert abs(t_x - 2 / (1 + n2)) < 0.01 * 2 / (1 + n2)

## 4. Verify

The checks of section 3 as asserts, against the closed forms.

- Speed at $S = 0.5$ within $0.2\,\%$ of $c$.
- Fresnel $r$ and $t$ at the $n = 2$ step within $1\,\%$.
- Energy: reflected plus transmitted equals incident within $1\,\%$.
- Antireflection: the reflected power at the design frequency below $2\,\%$ of the bare interface's. A perfect layer would give zero. The tolerance allows for the two things a grid cannot do exactly: the layer thickness is a whole number of cells, and numerical dispersion shifts the wavelength slightly, so the true null can sit a little off $f_0$ and the reflectance at $f_0$ is then small but not zero. The measured ratio is far below the tolerance.
- Lossy slab: the fitted attenuation within $1\,\%$ of the exact formula.

In [ ]:
checks = {
    "speed / c at S = 0.5": (speed[0.5], 1.0, 2e-3),
    "r at the n = 2 step": (r_meas, -1 / 3, 1e-2),
    "t at the n = 2 step": (t_meas, 2 / 3, 1e-2),
    "(U_ref + U_tr) / U_inc": ((U_ref + U_tr) / U_inc, 1.0, 1e-2),
    "alpha_fit / alpha": (alpha_fit / alpha, 1.0, 1e-2),
}
for name, (value, expect, tol) in checks.items():
    rel = abs(value / expect - 1)
    print(f"{name:26s} {value:+.6f}  expected {expect:+.6f}  relative error {rel:.1e}")
    assert rel < tol, name
ar = abs(r_coat[k0]) ** 2 / abs(r_bare[k0]) ** 2
print(f"{'R_coated / R_bare at f0':26s} {ar:.1e}  must be below 2e-2")
assert ar < 2e-2

## 5. Explore and rebuild

Explore. A slab of $\epsilon_r = 4$ and thickness 100 cells, with vacuum on both sides, is a Fabry-Perot etalon: the waves bouncing between its two faces add up, and the transmission depends on frequency. Send one broadband Gaussian pulse at it, record the transmitted trace behind the slab and the incident trace in front of it, and divide their Fourier transforms to get $|t(f)|^2$ over a wide band in a single run. Compare with the Airy function
$$T(f) = \frac{1}{1 + F\sin^2(2\pi n d f/c)}, \qquad F = \frac{4R}{(1 - R)^2}, \quad R = \left(\frac{n - 1}{n + 1}\right)^2.$$
Where do the transmission peaks sit, and why is the slab perfectly transparent there? How long must the run be for the peaks to be resolved, and how many cells per wavelength must the pulse spectrum keep before numerical dispersion moves the peaks visibly?

Rebuild. Close this notebook. In a blank editor, write the 1D Yee loop with arrays for $\epsilon_r$ and $\sigma$, a soft source and Mur boundaries at both ends, and reproduce the reflection at the $n = 2$ step and the speed at $S = 0.5$. Then set up the one-way initial pulse from memory, including the half-step shift of $H_z$.